# Creating a DAS Datastore from File Structure

This notebook demonstrates how to create a DAS datastore using the glob package to read files from a directory structure.

## Overview
- Use glob to discover DAS files in a directory
- Create a DASdatastore from the discovered files
- Explore the datastore structure

In [ ]:
import glob
import os
from pathlib import Path
import h5py
import numpy as np
from noisepy.io import H5DASdatastore
import matplotlib.pyplot as plt

## Step 1: Define data directory and search pattern

Modify the `data_dir` variable to point to your DAS data directory.

In [ ]:
# Define the directory containing DAS files
# Update this path to point to your actual DAS data
data_dir = "./sample_das_data"

# Define file patterns for different DAS file formats
# Common patterns: *.h5, *.hdf5, *.sgy, *.segy
file_patterns = [
    "*.h5",
    "*.hdf5",
    "*.sgy",
    "*.segy"
]

print(f"Looking for DAS files in: {data_dir}")
print(f"File patterns: {file_patterns}")

## Step 2: Use glob to discover DAS files

In [ ]:
def find_das_files(data_dir, patterns, recursive=True):
    """
    Find DAS files using glob patterns.
    
    Parameters:
    -----------
    data_dir : str
        Directory to search for files
    patterns : list
        List of glob patterns to match
    recursive : bool
        Whether to search recursively in subdirectories
    
    Returns:
    --------
    list
        List of found files
    """
    found_files = []
    
    for pattern in patterns:
        if recursive:
            search_pattern = os.path.join(data_dir, "**", pattern)
            files = glob.glob(search_pattern, recursive=True)
        else:
            search_pattern = os.path.join(data_dir, pattern)
            files = glob.glob(search_pattern)
        
        found_files.extend(files)
        print(f"Pattern '{pattern}': found {len(files)} files")
    
    return sorted(list(set(found_files)))  # Remove duplicates and sort

# Find DAS files
das_files = find_das_files(data_dir, file_patterns)

print(f"\nTotal DAS files found: {len(das_files)}")
if das_files:
    print("\nFirst few files:")
    for i, file in enumerate(das_files[:5]):
        print(f"  {i+1}: {file}")
else:
    print("\nNo DAS files found. Creating sample data for demonstration...")

## Step 3: Create sample data if no files are found

For demonstration purposes, we'll create some sample DAS data if no files are found.

In [ ]:
def create_sample_das_data(data_dir, num_files=3):
    """
    Create sample DAS data files for demonstration.
    
    Parameters:
    -----------
    data_dir : str
        Directory to create sample files in
    num_files : int
        Number of sample files to create
    """
    os.makedirs(data_dir, exist_ok=True)
    
    # Parameters for synthetic DAS data
    n_channels = 100  # Number of DAS channels
    n_samples = 1000  # Number of time samples
    sampling_rate = 100  # Hz
    
    created_files = []
    
    for i in range(num_files):
        filename = os.path.join(data_dir, f"sample_das_{i:03d}.h5")
        
        # Create synthetic DAS data (random noise + some coherent signals)
        time_axis = np.arange(n_samples) / sampling_rate
        
        # Add some synthetic signals
        signal = np.random.randn(n_channels, n_samples) * 0.1  # Background noise
        
        # Add a coherent signal across channels
        coherent_freq = 2.0  # Hz
        coherent_signal = np.sin(2 * np.pi * coherent_freq * time_axis)
        signal += coherent_signal[None, :] * 0.5
        
        # Save as HDF5 file
        with h5py.File(filename, 'w') as f:
            f.create_dataset('data', data=signal)
            f.create_dataset('time', data=time_axis)
            f.attrs['sampling_rate'] = sampling_rate
            f.attrs['n_channels'] = n_channels
            f.attrs['file_index'] = i
        
        created_files.append(filename)
        print(f"Created: {filename}")
    
    return created_files

# Create sample data if no files were found
if not das_files:
    print("Creating sample DAS data...")
    sample_files = create_sample_das_data(data_dir)
    
    # Re-run file discovery
    das_files = find_das_files(data_dir, file_patterns)
    print(f"\nAfter creating samples, found {len(das_files)} files")

## Step 4: Create DAS Datastore

Now we'll create a DAS datastore using the discovered files.

In [ ]:
# Create the datastore path
datastore_path = "./das_datastore.h5"

print(f"Creating DAS datastore: {datastore_path}")
print(f"Number of input files: {len(das_files)}")

try:
    # Initialize the H5DASdatastore
    # Note: The exact initialization may depend on the noisepy-io version
    # This is a template that should be adapted based on the actual API
    
    # For demonstration, we'll show the concept
    print("\nDatastore creation would involve:")
    print("1. Reading each DAS file")
    print("2. Extracting metadata (sampling rate, channels, etc.)")
    print("3. Organizing data in the H5DASdatastore format")
    print("4. Creating index structures for efficient access")
    
    # Example of what the datastore creation might look like:
    # datastore = H5DASdatastore(datastore_path)
    # for file in das_files:
    #     datastore.add_file(file)
    # datastore.finalize()
    
    print(f"\nDatastore would be saved as: {datastore_path}")
    
except Exception as e:
    print(f"Error creating datastore: {e}")
    print("This might be due to the specific noisepy-io API requirements.")
    print("Please refer to the noisepy-io documentation for the exact usage.")

## Step 5: Explore discovered files structure

Let's examine the structure of the discovered DAS files.

In [ ]:
def analyze_das_file(filepath):
    """
    Analyze a DAS file to extract basic information.
    
    Parameters:
    -----------
    filepath : str
        Path to the DAS file
    """
    try:
        if filepath.endswith(('.h5', '.hdf5')):
            with h5py.File(filepath, 'r') as f:
                print(f"\nFile: {os.path.basename(filepath)}")
                print(f"  Keys: {list(f.keys())}")
                
                if 'data' in f:
                    data_shape = f['data'].shape
                    print(f"  Data shape: {data_shape}")
                    print(f"  Data type: {f['data'].dtype}")
                
                # Print attributes
                if f.attrs:
                    print("  Attributes:")
                    for key, value in f.attrs.items():
                        print(f"    {key}: {value}")
        else:
            print(f"\nFile: {os.path.basename(filepath)} (format not supported for detailed analysis)")
            
    except Exception as e:
        print(f"Error analyzing {filepath}: {e}")

# Analyze the first few files
print("Analyzing DAS file structure:")
for file in das_files[:3]:  # Analyze first 3 files
    analyze_das_file(file)

## Step 6: Visualize file organization

Create a simple visualization of the discovered files.

In [ ]:
# Create a visualization of the file structure
if das_files:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot 1: Files by type
    file_extensions = {}
    for file in das_files:
        ext = Path(file).suffix.lower()
        file_extensions[ext] = file_extensions.get(ext, 0) + 1
    
    if file_extensions:
        exts, counts = zip(*file_extensions.items())
        ax1.bar(exts, counts)
        ax1.set_title('DAS Files by Type')
        ax1.set_xlabel('File Extension')
        ax1.set_ylabel('Number of Files')
    
    # Plot 2: File sizes (if we can get them)
    file_sizes = []
    file_names = []
    
    for file in das_files[:10]:  # First 10 files
        try:
            size = os.path.getsize(file) / (1024 * 1024)  # Convert to MB
            file_sizes.append(size)
            file_names.append(os.path.basename(file)[:15])  # Truncate long names
        except:
            continue
    
    if file_sizes:
        ax2.bar(range(len(file_sizes)), file_sizes)
        ax2.set_title('File Sizes (First 10 files)')
        ax2.set_xlabel('File')
        ax2.set_ylabel('Size (MB)')
        ax2.set_xticks(range(len(file_names)))
        ax2.set_xticklabels(file_names, rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nSummary:")
    print(f"Total files discovered: {len(das_files)}")
    print(f"File types: {list(file_extensions.keys())}")
    total_size = sum([os.path.getsize(f) for f in das_files if os.path.exists(f)]) / (1024**3)
    print(f"Total data size: {total_size:.2f} GB")
else:
    print("No files to visualize")

## Summary

This notebook demonstrated:

1. **File Discovery**: Using glob patterns to find DAS files in directory structures
2. **Data Organization**: Understanding the structure of discovered DAS files
3. **Datastore Concept**: The framework for creating a DAS datastore (actual implementation depends on noisepy-io API)

## Next Steps

- Adapt the datastore creation code to match the specific noisepy-io API
- Add error handling for different DAS file formats
- Implement metadata extraction for better file organization
- Add support for time-based file filtering and organization

See the next notebook for cross-correlation analysis using the created datastore.